In [1]:
import os
import numpy as np
import pandas as pd
import sys


In [2]:
print(pd.__version__)
print(np.__version__)

2.3.3
1.26.4


In [3]:
print(sys.executable)

C:\Users\PANKAJ SHARMA\anaconda3\envs\ml_clean\python.exe


In [4]:
df = pd.read_csv(r"D:\desktop\ML\Machine Learning\Supervised ML\Naive bayes\SMSSpamCollection", sep='\t'
                     )
df

,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
0,ham,Ok lar... Joking wif u oni...
1,spam,Free entry in 2 a wkly comp to win FA Cup fina...
2,ham,U dun say so early hor... U c already then say...
3,ham,"Nah I don't think he goes to usf, he lives aro..."
4,spam,FreeMsg Hey there darling it's been 3 week's n...
...,...,...
5566,spam,This is the 2nd time we have tried 2 contact u...
5567,ham,Will ü b going to esplanade fr home?
5568,ham,"Pity, * was in mood for that. So...any other s..."
5569,ham,The guy did some bitching but I acted like i'd...


In [5]:
df = pd.read_csv(r"D:\desktop\ML\Machine Learning\Supervised ML\Naive bayes\SMSSpamCollection", sep='\t',
                      names =["Label", "Messages"])
df

,Label,Messages
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [6]:
df.shape

(5572, 2)

In [7]:
df.isnull().sum()

Label       0
Messages    0
dtype: int64

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Label     5572 non-null   object
 1   Messages  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [9]:
df['Label'].unique()

array(['ham', 'spam'], dtype=object)

In [10]:
#df['Label'].value_counts()/len(df)*100
df['Label'].value_counts()

Label
ham     4825
spam     747
Name: count, dtype: int64

In [11]:
ham = df[df['Label']=='ham']
spam = df[df['Label']=='spam']

In [12]:
print(ham.shape, spam.shape)

(4825, 2) (747, 2)


In [13]:
# Upsample minority class, This is oversampling the minority class
spam = spam.sample(ham.shape[0], replace=True)

In [14]:
print(ham.shape, spam.shape) 

(4825, 2) (4825, 2)


In [15]:
data = pd.concat([ham, spam], ignore_index=True)
data.shape

(9650, 2)

# Shuffle After Concatenation (Very Important)
- Right now:
- All ham rows first
- All spam rows later
- Model may see structured ordering.

In [16]:
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
# .sample()? It randomly selects rows from a DataFrame.
# frac=1 means take 100% of rows,but in random order

In [17]:
data.tail()

,Label,Messages
9645,spam,Ur cash-balance is currently 500 pounds - to m...
9646,spam,URGENT!! Your 4* Costa Del Sol Holiday or £500...
9647,spam,Well done ENGLAND! Get the official poly ringt...
9648,ham,Just buy a pizza. Meat lovers or supreme. U ge...
9649,spam,WELL DONE! Your 4* Costa Del Sol Holiday or £5...


In [18]:
data['Label'].value_counts()

Label
ham     4825
spam    4825
Name: count, dtype: int64

# Data Cleaning part

In [19]:
# !pip install nltk
import re
import nltk
nltk.download('stopwords')
# stopwords: nltk.download('stopwords')


[nltk_data] Downloading package stopwords to C:\Users\PANKAJ
[nltk_data]     SHARMA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [21]:
ps = PorterStemmer()

stop_words = set(stopwords.words('english'))
print(stop_words)

{'my', 'having', "you've", 'only', "they'd", 'up', 'under', 'same', 'hers', 'you', 'mustn', 'haven', 'and', 'does', "i'll", 'no', 'hadn', "should've", 'above', "aren't", 'themselves', "shan't", "they're", 'wasn', 'had', 'now', "you'd", "it's", "i've", 'in', 'ain', 'this', 'those', 'after', "didn't", 'yourselves', 'against', 'mightn', 'because', 'couldn', 'y', 'more', "it'd", 'don', 'below', 'before', 'been', 't', "won't", 'out', 'they', 'm', "we've", "that'll", 'be', 'i', "couldn't", 'into', 'few', 'all', 'were', 'him', 'when', 'own', 'yourself', 'it', 'theirs', "you'll", "mightn't", 've', "it'll", "don't", 'is', 'such', 'she', 'what', 'hasn', 'ma', "he's", "she's", 'through', "she'll", 'once', 'won', 'on', "i'm", 'most', 'to', 'ours', 'that', 'he', 'the', "needn't", 'why', 'have', 'other', 'do', 'any', "hadn't", 'whom', 'where', 'these', 'being', 'will', 'during', 'a', 'from', 'did', "haven't", 'itself', 're', "they've", 'too', 'shouldn', "shouldn't", 'its', 'our', 'who', 'until', 'th

In [22]:
corpus = []

for text in data['Messages']:
    review = re.sub('[^a-zA-Z]',' ', text)
    review = review.lower()
    review = review.split()
    review = [ps.stem(word) for word in review if not word in stop_words]
    review = ' '.join(review)
    corpus.append(review)

# ps = PorterStemmer():  creates a stemming object used to reduce words to their root form to reduce vocabulary size and improve generalization.

| Original Word | Stemmed Output |
| ------------- | -------------- |
| playing       | play           |
| played        | play           |
| plays         | play           |
| studies       | studi          |
| running       | run            |


In [23]:
print(corpus)

['rememb', 'life mean lot love life love peopl life world call friend call world ge', 'e admin build might b slightli earlier call u reach', 'sorri went bed earli nightnight', 'import custom servic announc premier', 'free rington repli real', 'guarante latest nokia phone gb ipod mp player prize txt word collect tc llc ny usa p mt msgrcvd', 'nice day impress sensibl went home earli feel fine bore rememb', 'marvel mobil play offici ultim spider man game ur mobil right text spider game send u free ball wallpap', 'kindli send one flat lt decim gt today', 'hey darlin pick u colleg u tell wen mt love pete xx', 'collect valentin weekend pari inc flight hotel prize guarante text pari www rtf sphost com', 'fyi gonna call sporad start like lt gt bc doin shit', 'realli tot ur paper end long ago wat u copi ju got use u happi lar still haf studi', 'know pl open back', 'wake sinc check stuff saw true avail space pl call embassi send mail', 'jokin lar depend phone father get lor', 'free msg rington h

In [24]:
len(corpus)

9650

In [25]:
len(corpus[1000])

76

# Bag of words

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
#x_bow = cv.fit_transform(corpus).toarray()



In [27]:
x_bow = cv.fit_transform(corpus)

In [28]:
x_bow.shape

(9650, 6296)

In [29]:
type(x_bow)

scipy.sparse._csr.csr_matrix

In [30]:
x_bow[500].shape
# length of every row
# 6295 is vocabulary size

(1, 6296)

In [31]:
#print(x_bow)

In [32]:
data.head()

,Label,Messages
0,ham,So what about you. What do you remember
1,ham,"My life Means a lot to me, Not because I love ..."
2,ham,E admin building there? I might b slightly ear...
3,ham,"Sorry, went to bed early, nightnight"
4,spam,You have an important customer service announc...


In [33]:
data['Label'] = data['Label'].astype('category')
data['Label'] = data['Label'].cat.codes

In [34]:
data.shape

(9650, 2)

In [35]:
data[['Label']]

,Label
0,0
1,0
2,0
3,0
4,1
...,...
9645,1
9646,1
9647,1
9648,0


In [36]:
# split the data into training and test
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_bow, data['Label'], test_size=0.30, random_state=42,stratify = data['Label'])


In [37]:
#print(x_train)

In [38]:
x_test

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 30584 stored elements and shape (2895, 6296)>

In [39]:
y_train


2838    1
4160    1
4112    1
6674    1
6540    1
       ..
6959    1
6731    0
983     0
532     0
7775    0
Name: Label, Length: 6755, dtype: int8

In [40]:
y_test

5427    1
2164    0
8047    0
8586    1
1484    0
       ..
6196    1
2782    1
8644    0
2386    1
991     0
Name: Label, Length: 2895, dtype: int8

# Building Niave Bayes Theorem

In [41]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB

param_grid = {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0]}

nb_model_grid = GridSearchCV(MultinomialNB(), param_grid,scoring='f1', cv=5)
nb_model_grid.fit(x_train, y_train)

print(nb_model_grid.best_params_)

{'alpha': 0.01}


In [42]:
best_model = nb_model_grid.best_estimator_

In [43]:
y_pred_train = best_model.predict(x_train)
y_pred_test = best_model.predict(x_test)

In [44]:
y_pred_test

array([1, 0, 0, ..., 0, 1, 0], dtype=int8)

In [45]:
y_pred_train

array([1, 1, 1, ..., 0, 0, 0], dtype=int8)

# Evaluation

In [46]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [47]:
print(confusion_matrix(y_train, y_pred_train))
print()
print(confusion_matrix(y_test, y_pred_test))

[[3317   60]
 [  21 3357]]

[[1417   31]
 [  10 1437]]


In [48]:
print(classification_report(y_train, y_pred_train))
print()
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3377
           1       0.98      0.99      0.99      3378

    accuracy                           0.99      6755
   macro avg       0.99      0.99      0.99      6755
weighted avg       0.99      0.99      0.99      6755


              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1448
           1       0.98      0.99      0.99      1447

    accuracy                           0.99      2895
   macro avg       0.99      0.99      0.99      2895
weighted avg       0.99      0.99      0.99      2895



In [49]:
print(accuracy_score(y_train, y_pred_train))
print()
print(accuracy_score(y_test, y_pred_test))

0.9880088823094004

0.9858376511226252


# Step 1 — After Training, Create Prediction Function

In [50]:
import re

def preprocess_text(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    text = text.split()
    text = ' '.join(text)
    return text

# Step 2 — Predict New Email

In [51]:
def predict_email(email_text):
    # Step 1: Clean
    cleaned_text = preprocess_text(email_text)

    # Step 2: Vectorize (transform only!)
    vector = cv.transform([cleaned_text])

    # Step 3: Predict
    prediction = best_model.predict(vector)[0]

    if prediction == 1:
        return "Spam"
    else:
        return "Ham"

# Example Usage

In [55]:
new_mail = "Hello PANKAJ SHARMA,Thank you for registering for PC AIML Machine Learning. You can find information about this meeting below.PC AIML Machine Learning"

print(predict_email(new_mail))

Spam


In [53]:
new_mail = "Hello! Pankaj can you come tomorrow for urgent meeting"

print(predict_email(new_mail))

Ham


In [54]:
new_mail = 'urgent tri contact u today draw show prize guarante call land line claim c valid hr'

print(predict_email(new_mail))

Spam


| Step               | Manual Cleaning | Vectorizer     |
| ------------------ | --------------- | -------------- |
| Lowercase          | Yes             | Yes            |
| Remove punctuation | Yes             | Yes            |
| Remove stopwords   | Yes             | Yes (optional) |
| Stemming           | Yes             | No             |
| Needed for BOW?    | ❌ No            |                |
| Needed for TF-IDF? | ❌ No            |                |


# While need to save the model you should use concept of pipe line